In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import spatialdata as sd

In [ ]:
import scanpy as sc
import anndata as ad
from scipy import io
from scipy.sparse import coo_matrix, csr_matrix
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
import h5py
import squidpy as sq
import seaborn as sns
from shapely.geometry import Point, Polygon
from pathlib import Path
from typing import Dict, List, Tuple
import matplotlib.patches as patches

In [ ]:
import os
data_dir = "/saving_directory_name"  # Change this to your actual data directory
os.makedirs(data_dir, exist_ok=True)
os.chdir(data_dir)  

In [ ]:
adata = sc.read_h5ad('combined_all_datasets.h5ad') #h5ad file that after concat all tissues for this paper

In [ ]:
sc.pp.filter_cells(adata, min_counts=20)
sc.pp.filter_cells(adata, min_genes=10)
sc.pp.filter_genes(adata, min_cells=10)

In [ ]:
# Normalize total counts per cell (if not done)
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
# Find highly variable genes (HVGs)
sc.pp.highly_variable_genes(
    adata,
    flavor='cell_ranger',
    n_top_genes=2000,
    batch_key='batch',
)
adata.raw = adata.copy()
# Subset to HVGs only for downstream
adata = adata[:, adata.var.highly_variable]
sc.pp.regress_out(adata, ['total_counts'])
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, svd_solver='arpack')

In [ ]:
import harmonypy as hm
ho = hm.run_harmony(adata.obsm['X_pca'], adata.obs, 'batch')
adata.obsm['X_pca_harmony'] = ho.Z_corr.T

In [ ]:
sc.pp.neighbors(adata, n_neighbors=50, use_rep='X_pca_harmony')

In [ ]:
sc.tl.umap(adata)

In [ ]:
sc.tl.leiden(adata)

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden', method='wilcoxon')
sc.pl.rank_genes_groups(adata, n_genes=25, sharey=False)